In [1]:
from dataset import encode_in_text


/home/johannes/MTG_Popularity_Prediction/mtg_popularity_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 8951/8951 [00:03<00:00, 2930.88 examples/s]


In [18]:
print(encode_in_text("Lightning Bolt"))
print(encode_in_text(("Questing Beast")))

('Name: Lightning Bolt manaCost: red  type: Instant text: Lightning Bolt deals 3 damage to any target. ', 9.505320746909074)
("Name: Questing Beast manaCost: 2 generic mana green green  type: Legendary Creature — Beast power: 4 toughness: 4 text: Vigilance, deathtouch, haste Questing Beast can't be blocked by creatures with power 2 or less. Combat damage that would be dealt by creatures you control can't be prevented. Whenever Questing Beast deals combat damage to an opponent, it deals that much damage to target planeswalker that player controls. ", 4.2626798770413155)


In [11]:
f_card_1 = "Name: Strongest Beast manaCost: 2 generic mana green  type: Creature — Beast power: 6 toughness: 6 text: Vigilance, deathtouch, haste"
f_card_2 = "Name: Useless Knob, manaCost: 5 generic mana  type: Artifact Creature — Wall power: 0 toughness: 2 text: Defender"

In [6]:
from model import MTGPopularityPredictor

In [13]:
from transformers import RobertaTokenizer
import torch
from model import MTGPopularityPredictor
import numpy as np

class MTGPredictor:
    def __init__(self, model_path="best_mtg_predictor"):
        """
        Initialisiert den Predictor mit dem trainierten Modell.

        Args:
            model_path (str): Pfad zum gespeicherten Modell
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
        self.model = MTGPopularityPredictor.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()

    def preprocess_text(self, text):
        """
        Vorverarbeitung des Eingabetexts.

        Args:
            text (str): Eingabetext für die Vorhersage

        Returns:
            dict: Tokenisierter Text
        """
        return self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

    def predict(self, text):
        """
        Macht eine Vorhersage für einen gegebenen Kartentext.

        Args:
            text (str): Beschreibung der Karte im Format wie im Training

        Returns:
            float: Vorhergesagte Popularität (log-transformiert)
            float: Tatsächliche Anzahl (de-transformiert)
        """
        # Vorverarbeitung
        inputs = self.preprocess_text(text)

        # Verschiebe Input auf das richtige Device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Vorhersage
        with torch.no_grad():
            outputs = self.model(**inputs)
            prediction = outputs["logits"].cpu().numpy()[0]

        # De-transformiere die Vorhersage (log1p -> exp1m)
        actual_count = np.expm1(prediction)

        return prediction, actual_count

    def predict_batch(self, texts):
        """
        Macht Vorhersagen für mehrere Karten gleichzeitig.

        Args:
            texts (list): Liste von Kartentexten

        Returns:
            list: Liste von (log_prediction, actual_count) Tupeln
        """
        # Vorverarbeitung
        inputs = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        # Verschiebe Input auf das richtige Device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Vorhersage
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = outputs["logits"].cpu().numpy()

        # De-transformiere die Vorhersagen
        actual_counts = np.expm1(predictions)

        return list(zip(predictions, actual_counts))

In [26]:
predictor = MTGPredictor("best_mtg_predictor")
test_text_1 = 'Name: Divine Bolt manaCost: red  type: Instant text: Lightning Bolt deals 5 damage to any target. '
test_text_2 = "Name: Strongest Beast manaCost: 2 generic mana green  type: Creature — Beast power: 6 toughness: 6 text: Vigilance, deathtouch, haste"
test_text_3 = "Name: Useless Knob, manaCost: 5 generic mana  type: Artifact Creature — Wall power: 0 toughness: 2 text: Defender"

for n, test_text in enumerate([test_text_1, test_text_2, test_text_3]):
    log_pred, actual_count = predictor.predict(test_text)

    print(f"Predicted Count for Card {n + 1}: {int(actual_count)}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Predicted Count for Card 1: 2578
Predicted Count for Card 2: 21
Predicted Count for Card 3: 1
